Q IN WINDOW TO QUIT

In [ ]:
import time, cv2, mss, numpy as np, onnxruntime as ort
MONITOR = 1
MODEL = 'src/efficientnetv2/onnx/best_efficientnet_v2_s_terraria.onnx'
CLASSES = ['Corruption', 'Crimson', 'Desert', 'Dungeon', 'Forest', 'Hallow', 'Hell', 'Jungle', 'Mushroom', 'Ocean', 'Snow', 'Space', 'Underground']
MEAN = np.array([0.1473, 0.1647, 0.2079], np.float32); STD = np.array([0.1967, 0.2150, 0.2937], np.float32)
sess = ort.InferenceSession(MODEL, providers=[p for p in ['CUDAExecutionProvider', 'CPUExecutionProvider'] if p in ort.get_available_providers()])
inp, out = sess.get_inputs()[0].name, sess.get_outputs()[0].name

def prep(frame):
    x = cv2.resize(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB), (384, 216)).astype(np.float32) / 255
    return (((x - MEAN) / STD).transpose(2, 0, 1))[None]

cv2.namedWindow('Biome', cv2.WINDOW_NORMAL)
with mss.mss() as sct:
    try:
        while True:
            frame = np.array(sct.grab(sct.monitors[MONITOR]))[:, :, :3].copy()
            logits = sess.run([out], {inp: prep(frame)})[0][0]
            probs = np.exp(logits - logits.max()); probs /= probs.sum()
            i = probs.argmax()
            cv2.putText(frame, f'{CLASSES[i]} ({probs[i]:.3f})', (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            cv2.imshow('Biome', frame)
            if cv2.waitKey(1) & 255 == ord('q'): break
            time.sleep(0.03)
    finally:
        cv2.destroyAllWindows()
